# SMAP Data Extractor for Region of Interest

This notebook extracts soil moisture data from SMAP HDF5 files for a specific region of interest (ROI) in India.

**ROI Definition:**
- Bounding Box: (75, 30, 76, 31)
- WKT Polygon: POLYGON((75 30, 76 30, 76 31, 75 31, 75 30))

**Data Sources:**
1. SPL2SMAP_S - AMSR-2 Level 2 Land Data
2. SPL2SMP - SMAP L2 Radiometer Half-Orbit Soil Moisture
3. SPL2SMP_E - SMAP L2 Enhanced Radiometer Soil Moisture

**Year:** 2017 only

## Import Required Libraries

In [1]:
import h5py
import numpy as np
import pandas as pd
import xarray as xr
from datetime import datetime
import glob
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
os.chdir("/home/intern1/my_data/sm2rain-irrigation")

print("Libraries imported successfully!")

Libraries imported successfully!


## Define ROI and Configuration

In [2]:
# Region of Interest (ROI)
roi_bounds = {
    'min_lon': 75.0,
    'max_lon': 76.0,
    'min_lat': 30.0,
    'max_lat': 31.0
}

roi_wkt = "POLYGON((75 30, 76 30, 76 31, 75 31, 75 30))"

# Year filter - List of years to process
TARGET_YEARS = [2017,2018,2019,2020,2021]  # Add more years as needed, e.g., [2017, 2018, 2019]

# Data directories
base_dir = Path("/home/intern1/my_data/sm2rain-irrigation/NASA SMAP earthaccess/data")
data_dirs = {
    'SPL2SMAP_S': base_dir / 'SPL2SMAP_S',
    'SPL2SMP': base_dir / 'SPL2SMP',
    'SPL2SMP_E': base_dir / 'SPL2SMP_E'
}

# Output directory
output_dir = Path("/home/intern1/my_data/sm2rain-irrigation/NASA SMAP earthaccess/extracted_data_new")
output_dir.mkdir(exist_ok=True)

print("Configuration:")
print(f"  ROI: Lon [{roi_bounds['min_lon']}, {roi_bounds['max_lon']}], Lat [{roi_bounds['min_lat']}, {roi_bounds['max_lat']}]")
print(f"  Target Years: {TARGET_YEARS}")
print(f"  Output Directory: {output_dir}")

Configuration:
  ROI: Lon [75.0, 76.0], Lat [30.0, 31.0]
  Target Years: [2017, 2018, 2019, 2020, 2021]
  Output Directory: /home/intern1/my_data/sm2rain-irrigation/NASA SMAP earthaccess/extracted_data_new


## Helper Functions for Data Extraction

In [3]:
def is_within_roi(lat, lon, roi_bounds):
    """
    Check if coordinates are within ROI.
    Memory efficient - works with arrays.
    """
    return (
        (lon >= roi_bounds['min_lon']) & (lon <= roi_bounds['max_lon']) &
        (lat >= roi_bounds['min_lat']) & (lat <= roi_bounds['max_lat'])
    )

def extract_datetime_from_filename(filename):
    """
    Extract datetime from SMAP filename.
    Format examples:
    - SMAP_L2_SM_P_10259_D_20170102T010600_R19240_001.h5
    - AMSR_U2_L2_Land_B02_201701012024_D.he5
    """
    try:
        # For SMAP files (SPL2SMP, SPL2SMP_E)
        if 'SMAP' in filename:
            # Extract YYYYMMDDTHHMMSS
            parts = filename.split('_')
            for part in parts:
                if 'T' in part and len(part) >= 15:
                    date_str = part.split('T')[0]
                    time_str = part.split('T')[1][:6]
                    dt = datetime.strptime(date_str + time_str, '%Y%m%d%H%M%S')
                    return dt
        # For AMSR files (SPL2SMAP_S)
        elif 'AMSR' in filename:
            # Extract YYYYMMDDHHMM
            parts = filename.split('_')
            for part in parts:
                if len(part) == 12 and part.isdigit():
                    dt = datetime.strptime(part, '%Y%m%d%H%M')
                    return dt
    except Exception as e:
        print(f"Warning: Could not parse datetime from {filename}: {e}")
    return None

def extract_from_spl2smap_s(file_path, roi_bounds):
    """
    Extract data from SPL2SMAP_S (AMSR-2) HDF5 files.
    Returns DataFrame with lat, lon, SM, time.
    """
    try:
        with h5py.File(file_path, 'r') as f:
            # Access compound dataset
            data_path = 'HDFEOS/POINTS/AMSR-2 Level 2 Land Data/Data/Combined NPD and SCA Output Fields'
            compound_data = f[data_path][:]
            
            # Extract fields
            lat = compound_data['Latitude']
            lon = compound_data['Longitude']
            time = compound_data['Time']  # TAI93 time
            sm_npd = compound_data['SoilMoistureNPD']
            sm_sca = compound_data['SoilMoistureSCA']
            
            # Filter by ROI
            mask = is_within_roi(lat, lon, roi_bounds)
            
            if not np.any(mask):
                return None
            
            # Use SCA soil moisture (generally more accurate), fallback to NPD
            sm = np.where(sm_sca > 0, sm_sca, sm_npd)
            
            # Filter invalid values (typically < 0 or > 1)
            valid_mask = mask & (sm >= 0) & (sm <= 1)
            
            if not np.any(valid_mask):
                return None
            
            # Convert TAI93 time to datetime (TAI93 = seconds since 1993-01-01 00:00:00 TAI)
            tai93_epoch = datetime(1993, 1, 1)
            timestamps = [tai93_epoch + pd.Timedelta(seconds=float(t)) for t in time[valid_mask]]
            
            # Create DataFrame
            df = pd.DataFrame({
                'latitude': lat[valid_mask],
                'longitude': lon[valid_mask],
                'soil_moisture': sm[valid_mask],
                'time': timestamps
            })
            
            return df
            
    except Exception as e:
        print(f"Error processing {file_path.name}: {e}")
        return None

def extract_from_spl2smp(file_path, roi_bounds):
    """
    Extract data from SPL2SMP HDF5 files.
    Returns DataFrame with lat, lon, SM, time.
    """
    try:
        with h5py.File(file_path, 'r') as f:
            data_group = 'Soil_Moisture_Retrieval_Data'
            
            # Extract coordinates and soil moisture
            lat = f[data_group]['latitude'][:]
            lon = f[data_group]['longitude'][:]
            sm = f[data_group]['soil_moisture'][:]
            
            # Get time from filename
            dt = extract_datetime_from_filename(file_path.name)
            
            # Filter by ROI
            mask = is_within_roi(lat, lon, roi_bounds)
            
            if not np.any(mask):
                return None
            
            # Filter invalid values
            valid_mask = mask & (sm >= 0) & (sm <= 1) & (~np.isnan(sm))
            
            if not np.any(valid_mask):
                return None
            
            # Create DataFrame
            df = pd.DataFrame({
                'latitude': lat[valid_mask],
                'longitude': lon[valid_mask],
                'soil_moisture': sm[valid_mask],
                'time': dt
            })
            
            return df
            
    except Exception as e:
        print(f"Error processing {file_path.name}: {e}")
        return None

def extract_from_spl2smp_e(file_path, roi_bounds):
    """
    Extract data from SPL2SMP_E (Enhanced) HDF5 files.
    Returns DataFrame with lat, lon, SM, time.
    """
    try:
        with h5py.File(file_path, 'r') as f:
            # Try both possible group names
            data_group = None
            for group_name in ['Soil_Moisture_Retrieval_Data', 'Soil_Moisture_Retrieval_Data_PM']:
                if group_name in f:
                    data_group = group_name
                    break
            
            if data_group is None:
                print(f"Warning: No soil moisture data group found in {file_path.name}")
                return None
            
            # Extract coordinates and soil moisture
            lat = f[data_group]['latitude'][:]
            lon = f[data_group]['longitude'][:]
            sm = f[data_group]['soil_moisture'][:]
            
            # Get time from filename
            dt = extract_datetime_from_filename(file_path.name)
            
            # Filter by ROI
            mask = is_within_roi(lat, lon, roi_bounds)
            
            if not np.any(mask):
                return None
            
            # Filter invalid values
            valid_mask = mask & (sm >= 0) & (sm <= 1) & (~np.isnan(sm))
            
            if not np.any(valid_mask):
                return None
            
            # Create DataFrame
            df = pd.DataFrame({
                'latitude': lat[valid_mask],
                'longitude': lon[valid_mask],
                'soil_moisture': sm[valid_mask],
                'time': dt
            })
            
            return df
            
    except Exception as e:
        print(f"Error processing {file_path.name}: {e}")
        return None

print("Helper functions defined successfully!")

Helper functions defined successfully!


In [4]:
def save_to_netcdf(df, output_path, dataset_name, year):
    """
    Save DataFrame to NetCDF format.
    Creates a CF-compliant NetCDF file with lat, lon, SM, time dimensions.
    """
    if df is None or len(df) == 0:
        print(f"No data to save for {dataset_name} - {year}")
        return None

    # Ensure output path is a string and delete if exists
    output_file = str(output_path)
    if os.path.exists(output_file):
        try:
            os.remove(output_file)
            print(f"Removed existing file: {output_file}")
        except Exception as e:
            print(f"Warning: Could not remove existing file {output_file}: {e}")

    print(f"\nSaving {dataset_name} ({year}) to NetCDF...")

    # Prepare data with proper types
    soil_vals = df['soil_moisture'].values.astype(np.float32)
    lat_vals = df['latitude'].values.astype(np.float32)
    lon_vals = df['longitude'].values.astype(np.float32)
    
    # Convert time to datetime64 for proper NetCDF encoding
    time_vals = pd.to_datetime(df['time']).values

    # Create Dataset with proper coordinate structure
    # Use 'obs' as dimension name for observations
    ds = xr.Dataset(
        data_vars={
            'soil_moisture': (['obs'], soil_vals, {
                'long_name': 'Soil Moisture',
                'units': 'cm3/cm3',
                'valid_min': np.float32(0.0),
                'valid_max': np.float32(1.0),
                'description': 'Volumetric soil moisture content'
            }),
            'latitude': (['obs'], lat_vals, {
                'long_name': 'Latitude',
                'units': 'degrees_north',
                'standard_name': 'latitude'
            }),
            'longitude': (['obs'], lon_vals, {
                'long_name': 'Longitude',
                'units': 'degrees_east',
                'standard_name': 'longitude'
            }),
            'time': (['obs'], time_vals, {
                'long_name': 'Time of Observation',
                'standard_name': 'time'
            })
        },
        attrs={
            'title': f'{dataset_name} Soil Moisture Data - ROI Subset',
            'institution': 'SMAP/AMSR-2 Data',
            'source': dataset_name,
            'history': f'Created on {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}',
            'roi': f"Lon: {roi_bounds['min_lon']}-{roi_bounds['max_lon']}, Lat: {roi_bounds['min_lat']}-{roi_bounds['max_lat']}",
            'year': str(year),
            'conventions': 'CF-1.7'
        }
    )

    # Encoding with compression
    encoding = {
        'soil_moisture': {
            'zlib': True, 
            'complevel': 4, 
            'dtype': 'float32', 
            '_FillValue': np.float32(-9999.0)
        },
        'latitude': {
            'zlib': True, 
            'complevel': 4, 
            'dtype': 'float32'
        },
        'longitude': {
            'zlib': True, 
            'complevel': 4, 
            'dtype': 'float32'
        },
        'time': {
            'units': 'seconds since 1970-01-01',
            'calendar': 'gregorian',
            'dtype': 'int64'
        }
    }

    # try:
    #     # Save with proper engine and format
    #     # ds.to_netcdf(
    #     #     output_file, 
    #     #     format='NETCDF4', 
    #     #     engine='netcdf4', 
    #     #     encoding=encoding
    #     # )
    #     # ds.close()  # Explicitly close dataset
        
    #     print(f"✓ Saved to: {output_file}")
    #     file_size = os.path.getsize(output_file) / (1024 * 1024)
    #     print(f"  File size: {file_size:.2f} MB")
        
    #     # Verify the file can be read
    #     try:
    #         # test_ds = xr.open_dataset(output_file, decode_times=True)
    #         print(f"  Verification: OK")
    #         print(f"    - Observations: {len(test_ds['obs'])}")
    #         print(f"    - Variables: {list(test_ds.data_vars)}")
    #         print(f"    - Time range: {pd.to_datetime(test_ds['time'].values).min()} to {pd.to_datetime(test_ds['time'].values).max()}")
    #         # test_ds.close()
    #     except Exception as e:
    #         print(f"  Warning: File saved but verification failed: {e}")
        
    #     return output_file
        
    # except Exception as e:
    #     print(f"Error saving {dataset_name}: {e}")
    #     import traceback
    #     traceback.print_exc()
        
    #     # Fallback: Save as CSV if NetCDF fails
    try:
        csv_file = output_file.replace('.nc', '.csv')
        print(f"Attempting to save as CSV instead: {csv_file}")
        df.to_csv(csv_file, index=False)
        print(f"✓ Saved as CSV: {csv_file}")
        return csv_file
    except Exception as e2:
        print(f"Final error: Could not save as CSV either: {e2}")
        return None


## Main Extraction Function

In [5]:
def process_dataset(dataset_name, data_dir, roi_bounds, target_year, extraction_func):
    """
    Process all files in a dataset directory.
    Memory efficient - processes one file at a time.
    
    Args:
        dataset_name: Name of dataset (e.g., 'SPL2SMAP_S')
        data_dir: Path to data directory
        roi_bounds: ROI boundary dictionary
        target_year: Year to filter (e.g., 2017)
        extraction_func: Function to extract data from files
    
    Returns:
        DataFrame with all extracted data
    """
    print(f"\n{'='*60}")
    print(f"Processing {dataset_name}")
    print(f"{'='*60}")
    
    # Get all HDF5 files
    if dataset_name == 'SPL2SMAP_S':
        file_pattern = "*.he5"
    else:
        file_pattern = "*.h5"
    
    all_files = sorted(data_dir.glob(file_pattern))
    print(f"Total files found: {len(all_files)}")
    
    # Filter files by year (from filename)
    year_files = []
    for f in all_files:
        dt = extract_datetime_from_filename(f.name)
        if dt and dt.year == target_year:
            year_files.append(f)
    
    print(f"Files from year {target_year}: {len(year_files)}")
    
    if len(year_files) == 0:
        print(f"No files found for year {target_year}")
        return None
    
    # Process files and collect data
    all_data = []
    processed_count = 0
    roi_count = 0
    
    for i, file_path in enumerate(year_files, 1):
        if i % 50 == 0:
            print(f"Processing file {i}/{len(year_files)}... (ROI hits: {roi_count})")
        
        df = extraction_func(file_path, roi_bounds)
        
        if df is not None and len(df) > 0:
            all_data.append(df)
            processed_count += 1
            roi_count += len(df)
    
    print(f"\nProcessing complete!")
    print(f"  Files processed successfully: {processed_count}/{len(year_files)}")
    print(f"  Total data points in ROI: {roi_count}")
    
    if len(all_data) == 0:
        print(f"No data found within ROI for {dataset_name}")
        return None
    
    # Concatenate all data
    print("Concatenating data...")
    final_df = pd.concat(all_data, ignore_index=True)
    
    # Sort by time
    final_df = final_df.sort_values('time').reset_index(drop=True)
    
    print(f"Final dataset shape: {final_df.shape}")
    print(f"Date range: {final_df['time'].min()} to {final_df['time'].max()}")
    
    return final_df

print("Main extraction function defined!")

Main extraction function defined!


## Process SPL2SMAP_S (AMSR-2) Data

In [6]:
# Main Execution Loop
# Iterates over years and datasets, processes them, and saves to NetCDF

datasets_config = [
    {'name': 'SPL2SMAP_S', 'dir': data_dirs['SPL2SMAP_S'], 'func': extract_from_spl2smap_s, 'file_suffix': 'AMSR2'},
    {'name': 'SPL2SMP', 'dir': data_dirs['SPL2SMP'], 'func': extract_from_spl2smp, 'file_suffix': 'SMAP_L2'},
    {'name': 'SPL2SMP_E', 'dir': data_dirs['SPL2SMP_E'], 'func': extract_from_spl2smp_e, 'file_suffix': 'SMAP_L2_Enhanced'}
]

saved_files_summary = []

for year in TARGET_YEARS:
    print(f"\n{'#'*80}")
    print(f"PROCESSING YEAR: {year}")
    print(f"{'#'*80}")
    
    for config in datasets_config:
        dataset_name = config['name']
        
        # Process dataset for this year
        df = process_dataset(
            dataset_name=dataset_name,
            data_dir=config['dir'],
            roi_bounds=roi_bounds,
            target_year=year,
            extraction_func=config['func']
        )
        
        if df is not None and len(df) > 0:
            # Generate filename with year
            filename = f"{dataset_name}_{config['file_suffix']}_ROI_{year}.nc"
            output_path = output_dir / filename
            
            # Save to NetCDF
            saved_path = save_to_netcdf(df, output_path, dataset_name, year)
            
            if saved_path is not None:
                saved_files_summary.append({
                    'Dataset': dataset_name,
                    'Year': year,
                    'Filename': Path(saved_path).name,
                    'Points': len(df),
                    'Date_Range': f"{df['time'].min().date()} to {df['time'].max().date()}",
                    'Status': 'Success'
                })
            else:
                saved_files_summary.append({
                    'Dataset': dataset_name,
                    'Year': year,
                    'Filename': filename,
                    'Points': len(df),
                    'Date_Range': f"{df['time'].min().date()} to {df['time'].max().date()}",
                    'Status': 'Save Failed'
                })
            
            # Clear memory
            del df
        else:
            print(f"Skipping save for {dataset_name} {year} (no data)")
            saved_files_summary.append({
                'Dataset': dataset_name,
                'Year': year,
                'Filename': 'N/A',
                'Points': 0,
                'Date_Range': 'N/A',
                'Status': 'No Data'
            })

print(f"\n{'='*80}")
print("Multi-year processing complete!")
print(f"{'='*80}")

# Print summary
summary_df = pd.DataFrame(saved_files_summary)
print("\nProcessing Summary:")
print(summary_df.to_string(index=False))

# Save summary to file
summary_file =  output_dir / f'processing_summary_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
# summary_df.to_csv(summary_file, index=False)
# print(f"\nSummary saved to: {summary_file}")



################################################################################
PROCESSING YEAR: 2017
################################################################################

Processing SPL2SMAP_S
Total files found: 3555
Files from year 2017: 479
Total files found: 3555
Files from year 2017: 479
Processing file 50/479... (ROI hits: 590)
Processing file 50/479... (ROI hits: 590)
Processing file 100/479... (ROI hits: 1178)
Processing file 100/479... (ROI hits: 1178)
Processing file 150/479... (ROI hits: 1737)
Processing file 150/479... (ROI hits: 1737)
Processing file 200/479... (ROI hits: 2331)
Processing file 200/479... (ROI hits: 2331)
Processing file 250/479... (ROI hits: 2886)
Processing file 250/479... (ROI hits: 2886)
Processing file 300/479... (ROI hits: 3471)
Processing file 300/479... (ROI hits: 3471)
Processing file 350/479... (ROI hits: 4072)
Processing file 350/479... (ROI hits: 4072)
Processing file 400/479... (ROI hits: 4678)
Processing file 400/479... (ROI hits

## Process SPL2SMP (SMAP L2 Radiometer) Data

## Process SPL2SMP_E (SMAP L2 Enhanced) Data

## Save Data to NetCDF Files

## Summary Statistics and Verification

In [7]:
print("="*70)
print("EXTRACTION SUMMARY")
print("="*70)

summary_data = []

for df, filename, name in datasets_config:
    if df is not None and len(df) > 0:
        summary_data.append({
            'Dataset': name,
            'Total Points': len(df),
            'Unique Dates': df['time'].nunique(),
            'Date Range': f"{df['time'].min().date()} to {df['time'].max().date()}",
            'Lat Range': f"{df['latitude'].min():.3f} to {df['latitude'].max():.3f}",
            'Lon Range': f"{df['longitude'].min():.3f} to {df['longitude'].max():.3f}",
            'SM Mean': f"{df['soil_moisture'].mean():.4f}",
            'SM Std': f"{df['soil_moisture'].std():.4f}",
            'SM Min': f"{df['soil_moisture'].min():.4f}",
            'SM Max': f"{df['soil_moisture'].max():.4f}"
        })
    else:
        summary_data.append({
            'Dataset': name,
            'Total Points': 0,
            'Unique Dates': 0,
            'Date Range': 'No data',
            'Lat Range': 'N/A',
            'Lon Range': 'N/A',
            'SM Mean': 'N/A',
            'SM Std': 'N/A',
            'SM Min': 'N/A',
            'SM Max': 'N/A'
        })

summary_df = pd.DataFrame(summary_data)
print("\n", summary_df.to_string(index=False))

print("\n" + "="*70)
print(f"Output directory: {output_dir.absolute()}")
print("Files saved:")
for df, filename, name in datasets_to_save:
    if df is not None and len(df) > 0:
        print(f"  ✓ {filename}")
    else:
        print(f"  ✗ {filename} (no data)")
print("="*70)

EXTRACTION SUMMARY


ValueError: too many values to unpack (expected 3)

## Quick Visualization (Optional)

In [ ]:
import matplotlib.pyplot as plt

# Create a figure with subplots for each dataset
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

datasets = [
    (df_spl2smap_s, 'SPL2SMAP_S (AMSR-2)', axes[0]),
    (df_spl2smp, 'SPL2SMP (SMAP L2)', axes[1]),
    (df_spl2smp_e, 'SPL2SMP_E (Enhanced)', axes[2])
]

for df, title, ax in datasets:
    if df is not None and len(df) > 0:
        # Scatter plot of spatial distribution colored by soil moisture
        scatter = ax.scatter(df['longitude'], df['latitude'], 
                           c=df['soil_moisture'], cmap='YlGnBu',
                           s=20, alpha=0.6, vmin=0, vmax=0.5)
        ax.set_xlabel('Longitude', fontsize=10)
        ax.set_ylabel('Latitude', fontsize=10)
        ax.set_title(f'{title}\n({len(df)} points)', fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.set_xlim(roi_bounds['min_lon'], roi_bounds['max_lon'])
        ax.set_ylim(roi_bounds['min_lat'], roi_bounds['max_lat'])
        
        # Add colorbar
        cbar = plt.colorbar(scatter, ax=ax)
        cbar.set_label('Soil Moisture (cm³/cm³)', fontsize=9)
    else:
        ax.text(0.5, 0.5, 'No data available', 
               ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title, fontsize=11)

plt.tight_layout()
plt.savefig(output_dir / 'spatial_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Visualization saved to: {output_dir / 'spatial_distribution.png'}")

## Time Series Visualization (Optional)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

datasets = [
    (df_spl2smap_s, 'SPL2SMAP_S (AMSR-2)', axes[0]),
    (df_spl2smp, 'SPL2SMP (SMAP L2)', axes[1]),
    (df_spl2smp_e, 'SPL2SMP_E (Enhanced)', axes[2])
]

for df, title, ax in datasets:
    if df is not None and len(df) > 0:
        # Group by day and calculate mean
        daily_avg = df.groupby(df['time'].dt.date)['soil_moisture'].agg(['mean', 'std', 'count']).reset_index()
        daily_avg['time'] = pd.to_datetime(daily_avg['time'])
        
        # Plot mean with error bars
        ax.errorbar(daily_avg['time'], daily_avg['mean'], 
                   yerr=daily_avg['std'], fmt='o-', 
                   markersize=3, linewidth=1, alpha=0.7,
                   capsize=2, label='Daily Mean ± Std')
        
        ax.set_xlabel('Date', fontsize=10)
        ax.set_ylabel('Soil Moisture (cm³/cm³)', fontsize=10)
        ax.set_title(f'{title} - Time Series (Daily Average)', fontsize=11)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=9)
        
        # Add count as text
        ax2 = ax.twinx()
        ax2.bar(daily_avg['time'], daily_avg['count'], 
               alpha=0.2, color='gray', width=1, label='# observations')
        ax2.set_ylabel('Number of Observations', fontsize=9, color='gray')
        ax2.tick_params(axis='y', labelcolor='gray')
        ax2.legend(fontsize=8, loc='upper right')
    else:
        ax.text(0.5, 0.5, 'No data available', 
               ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title, fontsize=11)

plt.tight_layout()
plt.savefig(output_dir / 'time_series.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Time series plot saved to: {output_dir / 'time_series.png'}")

## How to Read the Generated NetCDF Files

To read the generated NetCDF files later, use the following code:

```python
import xarray as xr

# Read a NetCDF file
ds = xr.open_dataset('./extracted_data/SPL2SMAP_S_AMSR2_ROI_2017.nc')

# Access data
print(ds)
print("\nSoil moisture data:", ds['soil_moisture'].values)
print("Latitudes:", ds['latitude'].values)
print("Longitudes:", ds['longitude'].values)
print("Times:", ds['time'].values)

# Convert to DataFrame
df = ds.to_dataframe().reset_index()

# Close file
ds.close()
```

### Output Files:
- **SPL2SMAP_S_AMSR2_ROI_2017.nc**: AMSR-2 Level 2 Land Data
- **SPL2SMP_SMAP_L2_ROI_2017.nc**: SMAP L2 Radiometer Half-Orbit Data
- **SPL2SMP_E_SMAP_L2_Enhanced_ROI_2017.nc**: SMAP L2 Enhanced Radiometer Data

### Data Structure:
Each NetCDF file contains:
- `latitude`: Latitude coordinates (degrees_north)
- `longitude`: Longitude coordinates (degrees_east)
- `soil_moisture`: Volumetric soil moisture (cm³/cm³)
- `time`: Observation timestamp

### Notes:
- All data is filtered for ROI: Lon [75, 76], Lat [30, 31]
- Only data from year 2017 is included
- Invalid soil moisture values (< 0 or > 1, NaN) are filtered out
- Data is stored in observation-based format (not gridded)

In [ ]:
import xarray as xr

# Read a NetCDF file
ds = xr.open_dataset('/home/intern1/my_data/sm2rain-irrigation/NASA SMAP earthaccess/extracted_data_new/SPL2SMAP_S_AMSR2_ROI_2017.nc')

# Access data
print(ds)
print("\nSoil moisture data:", ds['soil_moisture'].values)
print("Latitudes:", ds['latitude'].values)
print("Longitudes:", ds['longitude'].values)
print("Times:", ds['time'].values)

# Convert to DataFrame
df = ds.to_dataframe().reset_index()

# Close file
ds.close()